<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_basic/03_mnist_example.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

# MNIST 손글씨 숫자 인식 - 딥러닝 입문

MNIST는 0부터 9까지의 손글씨 숫자 이미지 데이터셋으로, 딥러닝을 시작하는 "Hello World" 같은 예제입니다.

이 노트북에서는:
1. MNIST 데이터를 불러오고 탐색합니다
2. 간단한 신경망을 구축합니다
3. 모델을 학습하고 평가합니다
4. 실제 예측을 시각화합니다

In [ ]:
# PyTorch 임포트 (Google Colab T4 환경)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

print(f"PyTorch 버전: {torch.__version__}")

# GPU 설정 (Google Colab T4 환경)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print(f"Using device: {device}")

# 시드 설정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 1. MNIST 데이터셋 불러오기

In [ ]:
# 데이터 변환 정의 (이미지를 텐서로 변환하고 정규화)
transform = transforms.Compose([
    transforms.ToTensor(),  # PIL 이미지를 텐서로 변환 (0-255 -> 0-1)
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST의 평균과 표준편차로 정규화
])

# 학습 데이터와 테스트 데이터 다운로드
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# DataLoader 생성 (배치 단위로 데이터를 불러옴)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"학습 데이터 개수: {len(train_dataset):,}")
print(f"테스트 데이터 개수: {len(test_dataset):,}")
print(f"이미지 크기: 28x28")
print(f"클래스 수: 10 (0-9)")

## 2. 데이터 탐색 및 시각화

In [ ]:
# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
axes = axes.ravel()

for i in range(10):
    # 데이터셋에서 이미지와 레이블 가져오기
    img, label = train_dataset[i]
    
    # 텐서를 numpy 배열로 변환하고 정규화 역변환
    img = img.squeeze().numpy()
    img = img * 0.3081 + 0.1307  # 정규화 역변환
    
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'Label: {label}', fontsize=12)
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('MNIST 샘플 이미지', fontsize=16, y=1.02)
plt.show()

In [ ]:
# 각 숫자별 샘플 개수 확인
train_labels = [train_dataset[i][1] for i in range(len(train_dataset))]
unique, counts = np.unique(train_labels, return_counts=True)

plt.figure(figsize=(10, 6))
plt.bar(unique, counts, color='skyblue', edgecolor='navy')
plt.xlabel('숫자 클래스', fontsize=12)
plt.ylabel('샘플 개수', fontsize=12)
plt.title('MNIST 학습 데이터의 클래스별 분포', fontsize=14)
plt.grid(axis='y', alpha=0.3)
for i, count in enumerate(counts):
    plt.text(i, count + 100, str(count), ha='center', fontsize=10)
plt.show()

## 3. 신경망 모델 구축

세 가지 다른 복잡도의 모델을 구현합니다:
1. **SimpleNet**: 가장 간단한 2층 신경망
2. **ImprovedNet**: 더 깊고 드롭아웃이 있는 신경망
3. **ConvNet**: 합성곱 신경망 (CNN)

In [ ]:
# 1. 간단한 신경망
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)  # 입력층 -> 은닉층
        self.fc2 = nn.Linear(128, 10)     # 은닉층 -> 출력층
        
    def forward(self, x):
        x = x.view(-1, 28*28)  # 이미지를 1차원으로 펼침
        x = F.relu(self.fc1(x))  # 활성화 함수
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)  # 로그 소프트맥스 (확률 분포)

# 2. 개선된 신경망
class ImprovedNet(nn.Module):
    def __init__(self):
        super(ImprovedNet, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(128, 64)
        self.dropout3 = nn.Dropout(0.2)
        self.fc4 = nn.Linear(64, 10)
        
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = F.relu(self.fc3(x))
        x = self.dropout3(x)
        x = self.fc4(x)
        return F.log_softmax(x, dim=1)

# 3. 합성곱 신경망 (CNN)
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        # 합성곱층
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout1 = nn.Dropout2d(0.25)
        
        # 완전연결층
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        # 합성곱 -> ReLU -> 풀링
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout1(x)
        
        # 텐서를 1차원으로 펼침
        x = x.view(-1, 64 * 7 * 7)
        
        # 완전연결층
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        
        return F.log_softmax(x, dim=1)

# 모델 생성
simple_model = SimpleNet().to(device)
improved_model = ImprovedNet().to(device)
conv_model = ConvNet().to(device)

print("모델이 생성되었습니다!")
print(f"\nSimpleNet 파라미터 수: {sum(p.numel() for p in simple_model.parameters()):,}")
print(f"ImprovedNet 파라미터 수: {sum(p.numel() for p in improved_model.parameters()):,}")
print(f"ConvNet 파라미터 수: {sum(p.numel() for p in conv_model.parameters()):,}")

## 4. 학습 및 평가 함수 정의

In [ ]:
def train_epoch(model, train_loader, optimizer, epoch):
    """
    한 에폭 동안 모델을 학습합니다.
    """
    model.train()
    train_loss = 0
    correct = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}')
    for batch_idx, (data, target) in enumerate(pbar):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        
        # 진행률 표시줄 업데이트
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100. * correct / ((batch_idx + 1) * batch_size):.2f}%'
        })
    
    avg_loss = train_loss / len(train_loader)
    accuracy = 100. * correct / len(train_loader.dataset)
    
    return avg_loss, accuracy

def evaluate(model, test_loader):
    """
    모델을 평가합니다.
    """
    model.eval()
    test_loss = 0
    correct = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    
    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    
    return test_loss, accuracy

def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    """
    모델을 학습하고 결과를 기록합니다.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, epoch)
        test_loss, test_acc = evaluate(model, test_loader)
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        print(f'\nEpoch {epoch}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, '
              f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%\n')
    
    return {
        'train_losses': train_losses,
        'test_losses': test_losses,
        'train_accs': train_accs,
        'test_accs': test_accs
    }

## 5. 간단한 모델 학습

In [ ]:
print("간단한 신경망 학습 시작...")
simple_results = train_model(simple_model, train_loader, test_loader, epochs=5)

## 6. 개선된 모델 학습

In [ ]:
print("개선된 신경망 학습 시작...")
improved_results = train_model(improved_model, train_loader, test_loader, epochs=5)

## 7. CNN 모델 학습

In [ ]:
print("CNN 학습 시작...")
conv_results = train_model(conv_model, train_loader, test_loader, epochs=5)

## 8. 모델 성능 비교

In [ ]:
# 학습 곡선 시각화
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))

# 학습 손실
ax1.plot(simple_results['train_losses'], label='SimpleNet', marker='o')
ax1.plot(improved_results['train_losses'], label='ImprovedNet', marker='s')
ax1.plot(conv_results['train_losses'], label='ConvNet', marker='^')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('학습 손실 변화')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 테스트 손실
ax2.plot(simple_results['test_losses'], label='SimpleNet', marker='o')
ax2.plot(improved_results['test_losses'], label='ImprovedNet', marker='s')
ax2.plot(conv_results['test_losses'], label='ConvNet', marker='^')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Loss')
ax2.set_title('테스트 손실 변화')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 학습 정확도
ax3.plot(simple_results['train_accs'], label='SimpleNet', marker='o')
ax3.plot(improved_results['train_accs'], label='ImprovedNet', marker='s')
ax3.plot(conv_results['train_accs'], label='ConvNet', marker='^')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Training Accuracy (%)')
ax3.set_title('학습 정확도 변화')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 테스트 정확도
ax4.plot(simple_results['test_accs'], label='SimpleNet', marker='o')
ax4.plot(improved_results['test_accs'], label='ImprovedNet', marker='s')
ax4.plot(conv_results['test_accs'], label='ConvNet', marker='^')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Test Accuracy (%)')
ax4.set_title('테스트 정확도 변화')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 최종 성능 비교
print("\n=== 최종 성능 비교 ===")
print(f"SimpleNet - Test Accuracy: {simple_results['test_accs'][-1]:.2f}%")
print(f"ImprovedNet - Test Accuracy: {improved_results['test_accs'][-1]:.2f}%")
print(f"ConvNet - Test Accuracy: {conv_results['test_accs'][-1]:.2f}%")

## 9. 실제 예측 시각화

In [ ]:
def visualize_predictions(model, test_loader, num_images=10):
    """
    모델의 예측 결과를 시각화합니다.
    """
    model.eval()
    images, labels = next(iter(test_loader))
    images, labels = images[:num_images].to(device), labels[:num_images]
    
    with torch.no_grad():
        outputs = model(images)
        predictions = outputs.argmax(dim=1).cpu()
        probabilities = torch.exp(outputs).cpu()  # log_softmax를 확률로 변환
    
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    axes = axes.ravel()
    
    for i in range(num_images):
        img = images[i].cpu().squeeze()
        img = img * 0.3081 + 0.1307  # 정규화 역변환
        
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        
        # 예측이 맞으면 파란색, 틀리면 빨간색
        color = 'blue' if predictions[i] == labels[i] else 'red'
        confidence = probabilities[i][predictions[i]].item() * 100
        
        axes[i].set_title(f'실제: {labels[i]}\n예측: {predictions[i]} ({confidence:.1f}%)', 
                         color=color, fontsize=10)
    
    plt.tight_layout()
    return fig

# 각 모델의 예측 시각화
print("SimpleNet 예측:")
fig1 = visualize_predictions(simple_model, test_loader)
plt.show()

print("\nImprovedNet 예측:")
fig2 = visualize_predictions(improved_model, test_loader)
plt.show()

print("\nConvNet 예측:")
fig3 = visualize_predictions(conv_model, test_loader)
plt.show()

## 10. 혼동 행렬 (Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def plot_confusion_matrix(model, test_loader, title):
    """
    모델의 혼동 행렬을 그립니다.
    """
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)
            output = model(data)
            predictions = output.argmax(dim=1).cpu().numpy()
            all_predictions.extend(predictions)
            all_labels.extend(target.numpy())
    
    cm = confusion_matrix(all_labels, all_predictions)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=range(10), yticklabels=range(10))
    plt.xlabel('예측된 클래스', fontsize=12)
    plt.ylabel('실제 클래스', fontsize=12)
    plt.title(title, fontsize=14)
    plt.show()
    
    # 클래스별 정확도 계산
    class_accuracy = cm.diagonal() / cm.sum(axis=1)
    for i in range(10):
        print(f"숫자 {i}의 정확도: {class_accuracy[i]*100:.2f}%")

# ConvNet의 혼동 행렬 시각화 (가장 성능이 좋은 모델)
plot_confusion_matrix(conv_model, test_loader, "ConvNet 혼동 행렬")

## 11. 틀린 예측 분석

In [ ]:
def analyze_mistakes(model, test_loader, num_mistakes=12):
    """
    모델이 틀린 예측을 분석합니다.
    """
    model.eval()
    mistakes = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target
            output = model(data)
            predictions = output.argmax(dim=1).cpu()
            probabilities = torch.exp(output).cpu()
            
            # 틀린 예측 찾기
            wrong_idx = (predictions != target).nonzero(as_tuple=True)[0]
            
            for idx in wrong_idx:
                mistakes.append({
                    'image': data[idx].cpu(),
                    'true_label': target[idx].item(),
                    'pred_label': predictions[idx].item(),
                    'confidence': probabilities[idx][predictions[idx]].item()
                })
                
                if len(mistakes) >= num_mistakes:
                    break
            
            if len(mistakes) >= num_mistakes:
                break
    
    # 틀린 예측 시각화
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    axes = axes.ravel()
    
    for i, mistake in enumerate(mistakes[:num_mistakes]):
        img = mistake['image'].squeeze()
        img = img * 0.3081 + 0.1307  # 정규화 역변환
        
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f"실제: {mistake['true_label']}\n"
                         f"예측: {mistake['pred_label']} "
                         f"({mistake['confidence']*100:.1f}%)",
                         color='red', fontsize=10)
    
    plt.suptitle('모델이 틀린 예측들', fontsize=16)
    plt.tight_layout()
    plt.show()

# ConvNet이 틀린 예측 분석
print("ConvNet이 틀린 예측들:")
analyze_mistakes(conv_model, test_loader)

## 12. 직접 그린 숫자 테스트

In [ ]:
import cv2
from PIL import Image

def create_digit_image():
    """
    간단한 숫자 이미지를 프로그래밍 방식으로 생성합니다.
    """
    # 28x28 검은 이미지 생성
    img = np.zeros((28, 28), dtype=np.uint8)
    
    # 숫자 7 그리기
    # 상단 가로선
    img[5:7, 8:20] = 255
    # 대각선
    for i in range(15):
        img[7+i, 18-i:20-i] = 255
    
    return img

def test_custom_digit(model, img_array):
    """
    사용자 정의 숫자 이미지를 테스트합니다.
    """
    # 전처리
    img_tensor = transform(Image.fromarray(img_array)).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.exp(output).cpu().numpy()[0]
        prediction = output.argmax(dim=1).item()
    
    # 시각화
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    # 이미지 표시
    ax1.imshow(img_array, cmap='gray')
    ax1.set_title(f'예측: {prediction} ({probabilities[prediction]*100:.1f}%)', fontsize=14)
    ax1.axis('off')
    
    # 확률 분포 표시
    ax2.bar(range(10), probabilities)
    ax2.set_xlabel('숫자', fontsize=12)
    ax2.set_ylabel('확률', fontsize=12)
    ax2.set_title('각 숫자에 대한 예측 확률', fontsize=14)
    ax2.set_xticks(range(10))
    
    plt.tight_layout()
    plt.show()

# 프로그래밍 방식으로 생성한 숫자 테스트
custom_digit = create_digit_image()
print("프로그래밍 방식으로 생성한 숫자 7 테스트:")
test_custom_digit(conv_model, custom_digit)

## 13. 모델 저장 및 불러오기

In [ ]:
# 모델 저장
torch.save(conv_model.state_dict(), 'mnist_conv_model.pth')
print("모델이 'mnist_conv_model.pth'로 저장되었습니다.")

# 모델 불러오기 예시
# loaded_model = ConvNet().to(device)
# loaded_model.load_state_dict(torch.load('mnist_conv_model.pth'))
# loaded_model.eval()
# print("모델을 성공적으로 불러왔습니다!")

## 요약

이 노트북에서는 MNIST 데이터셋을 사용하여 딥러닝의 기초를 학습했습니다:

1. **데이터 전처리**: 이미지를 텐서로 변환하고 정규화
2. **모델 구조**: 간단한 FC부터 CNN까지 다양한 모델 구현
3. **학습 과정**: 손실 함수, 옵티마이저, 배치 학습
4. **성능 평가**: 정확도, 혼동 행렬, 틀린 예측 분석

### 주요 관찰 사항:
- **SimpleNet** (2층): 기본적인 성능 (~97%)
- **ImprovedNet** (4층 + Dropout): 더 나은 일반화
- **ConvNet** (CNN): 최고의 성능 (~99%)

### 다음 단계:
- 데이터 증강 (회전, 이동 등) 적용
- 더 복잡한 CNN 구조 실험
- 다른 데이터셋 (Fashion-MNIST, CIFAR-10) 시도